Primero obtenemos la muestra, en este caso son 300000 filas del dataset de entrenamiento.

In [ ]:
import polars as pl

# Lazy scan: NO carga nada en memoria todavía
lf = pl.scan_csv("archive/data/splits/*.csv")

# Muestreo aleatorio directamente en disco
muestra = (
    lf.collect(streaming=True)      # procesa por chunks
      .sample(n=200_000, seed=42)
)

muestra.write_csv("archive/data/split/muestra_200k.csv")
print(muestra.shape)

/tmp/ipykernel_18005/4038530206.py:8: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  lf.collect(streaming=True)      # procesa por chunks


(200000, 46)


___
### Que modelo generativo usaremos?
Para la creación de datos sintenticos usaremos TVAE, en el estudio que deje en bibliografia explica que es el segundo de 3 meteodos de creacion de datos sinteticos que mejores datos le da, de esa forma podemos tener un resultado intermedio.
___
### Que es y como funciona TVAE?
TVAE es un modelo generativo creado en 2019, este autoencoder presenta una caracteristica clave y es que **no codifica cada punto en un lugar fijo, sino que usar una distribución gausseana para poder muestrear los puntos.**

In [1]:
import pandas as pd
from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer

# Cargar una muestra de los datos
real_data = pd.read_csv("archive/data/splits/muestra_200k.csv")

# Detectar automáticamente la metadata (tipos de columnas)
metadata = Metadata.detect_from_dataframe(
    data=real_data,
    table_name='mi_dataset'
)

In [1]:
import pandas as pd

df = pd.read_csv("archive/data/splits/muestra_200k.csv", nrows=200_000)

# 1. Tamaño real en RAM
print(f"RAM: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Shape: {df.shape}")

# 2. Esto es lo que más importa: cardinalidad de categóricas
print("\nColumnas categóricas y sus valores únicos:")
for col in df.columns:
    if df[col].dtype == 'object' or df[col].dtype.name == 'category':
        print(f"  {col}: {df[col].nunique()} únicos")
    elif df[col].nunique() < 50:
        print(f"  {col} (numérica con pocos valores): {df[col].nunique()} únicos")
        

RAM: 0.17 GB
Shape: (200000, 46)

Columnas categóricas y sus valores únicos:
  product_id: 9980 únicos
  title: 9825 únicos
  brand: 62 únicos
  category: 7 únicos
  subcategory: 35 únicos
  platform: 5 únicos
  timestamp: 730 únicos
  year (numérica con pocos valores): 2 únicos
  month (numérica con pocos valores): 12 únicos
  day (numérica con pocos valores): 31 únicos
  dayofweek (numérica con pocos valores): 7 únicos
  stock_status: 3 únicos
  avg_rating (numérica con pocos valores): 44 únicos
  is_flagship (numérica con pocos valores): 2 únicos
  launch_year (numérica con pocos valores): 7 únicos
  asin: 9980 únicos
  is_all_time_low (numérica con pocos valores): 2 únicos
  is_all_time_high (numérica con pocos valores): 2 únicos
  fake_discount_flag (numérica con pocos valores): 1 únicos
  price_spike_flag (numérica con pocos valores): 2 únicos
  is_black_friday_week (numérica con pocos valores): 2 únicos
  is_holiday_season (numérica con pocos valores): 2 únicos
  is_weekend (num

In [1]:
import pandas as pd

df = pd.read_csv("archive/data/splits/muestra_200k.csv")

print("=" * 60)
print("INFORMACIÓN BÁSICA")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"RAM: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print(f"Columnas: {len(df.columns)}")

print("\n" + "=" * 60)
print("TIPOS DE DATOS")
print("=" * 60)
print(df.dtypes)

print("\n" + "=" * 60)
print("VALORES ÚNICOS POR COLUMNA (ordenado de mayor a menor)")
print("=" * 60)
unicos = df.nunique().sort_values(ascending=False)
for col, n in unicos.items():
    flag = ""
    if n > 1000:
        flag = " 🔴 ALTA CARDINALIDAD"
    elif n > 100:
        flag = " 🟡 media-alta"
    print(f"  {col}: {n}{flag}")

print("\n" + "=" * 60)
print("COLUMNAS CON VALORES NULOS")
print("=" * 60)
nulos = df.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos) > 0:
    for col, n in nulos.items():
        print(f"  {col}: {n} ({n/len(df)*100:.1f}%)")
else:
    print("  Ninguna")

print("\n" + "=" * 60)
print("PRIMERAS 3 FILAS")
print("=" * 60)
print(df.head(3).to_string())

print("\n" + "=" * 60)
print("COLUMNAS DE TEXTO LIBRE (strings largos)")
print("=" * 60)
for col in df.select_dtypes(include=['object']).columns:
    sample = df[col].dropna().iloc[0] if len(df[col].dropna()) > 0 else ""
    max_len = df[col].astype(str).str.len().max()
    avg_len = df[col].astype(str).str.len().mean()
    print(f"  {col}: max_len={max_len}, avg_len={avg_len:.1f}")

INFORMACIÓN BÁSICA
Shape: (200000, 46)
RAM: 0.17 GB
Columnas: 46

TIPOS DE DATOS
product_id                    object
title                         object
brand                         object
category                      object
subcategory                   object
platform                      object
timestamp                     object
year                         float64
month                        float64
day                          float64
dayofweek                    float64
price                        float64
original_price               float64
discount_pct                 float64
stock_status                  object
avg_rating                   float64
review_count                 float64
is_flagship                  float64
launch_year                  float64
asin                          object
price_change_abs             float64
price_change_pct             float64
price_7d_avg                 float64
price_30d_avg                float64
price_7d_std                 fl

In [ ]:
# ============================================================
# generar_sinteticos.py — versión completa con todos los fixes
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

import time
import gc
import pandas as pd
import torch
torch.set_num_threads(2)

from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================
RUTA_CSV = "archive/data/splits/muestra_200k.csv"
N_FILAS_SINTETICAS = 1_000_00
ARCHIVO_SALIDA = "datos_sinteticos.csv"

# Columnas de alta cardinalidad → marcar como 'id' (no se modelan)
COLUMNAS_ID = ['product_id', 'title', 'asin']

# Columnas a eliminar (timestamp ya está cubierta por year/month/day)
COLUMNAS_A_ELIMINAR = ['timestamp']

# 🔥 FIX CLAVE: columnas numéricas que SDV detecta como categóricas
# por su altísima cardinalidad (30k-80k valores únicos)
COLUMNAS_NUMERICAS_FORZADAS = [
    'original_price', 'price_30d_max', 'price', 'price_7d_max',
    'price_30d_avg', 'price_7d_avg', 'target_price_7d', 'target_price_30d',
    'price_7d_min', 'price_30d_min', 'value_score', 'price_52w_min',
    'price_52w_max', 'price_30d_std', 'price_change_abs',
    'discount_vs_30d_avg', 'price_7d_std', 'popularity_score',
    'price_change_pct', 'price_volatility_30d', 'discount_pct',
    'review_count'
]


# ============================================================
# 2. LEER DATOS
# ============================================================
log("Leyendo datos...")
real_data = pd.read_csv(RUTA_CSV)
log(f"Shape inicial: {real_data.shape}")
log(f"RAM: {real_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")


# ============================================================
# 3. LIMPIAR
# ============================================================
for col in COLUMNAS_A_ELIMINAR:
    if col in real_data.columns:
        real_data = real_data.drop(columns=[col])
        log(f"Eliminada: {col}")

real_data = real_data.fillna(0)
log("Nulos rellenados con 0")

gc.collect()


# ============================================================
# 4. DETECTAR METADATA
# ============================================================
log("Detectando metadata...")
metadata = Metadata.detect_from_dataframe(real_data, table_name='mi_dataset')


# ============================================================
# 5. CORREGIR METADATA
# ============================================================
# 5a. Marcar IDs (el fix que ya tenías)
for col in COLUMNAS_ID:
    if col in real_data.columns:
        metadata.update_column(column_name=col, sdtype='id')
        log(f"'{col}' → id")

# 5b. 🔥 Forzar numéricas (EL FIX QUE FALTABA)
for col in COLUMNAS_NUMERICAS_FORZADAS:
    if col in real_data.columns:
        metadata.update_column(column_name=col, sdtype='numerical')
log(f"Forzadas {len(COLUMNAS_NUMERICAS_FORZADAS)} columnas como numéricas")


# ============================================================
# 6. CREAR Y ENTRENAR
# ============================================================
log("Creando sintetizador TVAE...")
synthesizer = TVAESynthesizer(
    metadata,
    epochs=50,
    batch_size=500,
    enforce_min_max_values=True,
    enforce_rounding=True,
    enable_gpu=torch.cuda.is_available(),
    verbose=True,
)

log(f"GPU disponible: {torch.cuda.is_available()}")
log("Iniciando fit...")
synthesizer.fit(real_data)
log("¡Fit terminado!")


# ============================================================
# 7. GENERAR
# ============================================================
log(f"Generando {N_FILAS_SINTETICAS} filas...")
synthetic_data = synthesizer.sample(num_rows=N_FILAS_SINTETICAS)
log(f"Sintético: {synthetic_data.shape}")


# ============================================================
# 8. GUARDAR
# ============================================================
synthetic_data.to_csv(ARCHIVO_SALIDA, index=False)
real_data.to_csv("datos_reales_usados.csv", index=False)
log(f"Guardados: {ARCHIVO_SALIDA} y datos_reales_usados.csv")

[16:15:53] Leyendo datos...
[16:15:54] Shape inicial: (200000, 46)
[16:15:54] RAM: 0.17 GB
[16:15:54] Eliminada: timestamp
[16:15:54] Nulos rellenados con 0
[16:15:54] Detectando metadata...
[16:15:55] 'product_id' → id
[16:15:55] 'title' → id
[16:15:55] 'asin' → id
[16:15:55] Forzadas 22 columnas como numéricas
[16:15:55] Creando sintetizador TVAE...
[16:15:55] GPU disponible: False
[16:15:55] Iniciando fit...


In [ ]:
# Crear el sintetizador TVAE
synthesizer = TVAESynthesizer(
    metadata,
    epochs=30,                   
    enforce_min_max_values=True,  
    enforce_rounding=True,     
    verbose= True    
)

# Entrenar el modelo
print("Entrenando TVAE...")
synthesizer.fit(real_data)
print("¡Entrenamiento completado!")

Entrenando TVAE...


In [ ]:
# Generar las muestras sintéticas
synthetic_data = synthesizer.sample(num_rows=200_000)

# Guardar para usar después
synthetic_data.to_csv("datos_sinteticos.csv", index=False)
print(f"Datos sintéticos generados: {synthetic_data.shape}")

In [1]:
import pandas as pd

df= pd.read_csv("datos_sinteticos_copula.csv")

print(f"filas datos sinteticos: {len(df)}")

filas datos sinteticos: 200000


## Cambio de planes

TVAE tiro muchos errores y por simplicidad se usara gaussiancopula que es el más fácil

In [ ]:
# ============================================================
# generar_sinteticos_copula.py
# Genera datos sintéticos con GaussianCopula (rápido y ligero)
# ============================================================
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

import time
import gc
import pandas as pd

from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer


def log(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


# ============================================================
# 1. CONFIGURACIÓN
# ============================================================
RUTA_CSV = "archive/data/splits/muestra_200k.csv"
N_FILAS_SINTETICAS = 5_000_000
ARCHIVO_SALIDA = "datos_sinteticos_copula.csv"

COLUMNAS_ID = ['product_id', 'title', 'asin']
COLUMNAS_A_ELIMINAR = ['timestamp']

COLUMNAS_NUMERICAS_FORZADAS = [
    'original_price', 'price_30d_max', 'price', 'price_7d_max',
    'price_30d_avg', 'price_7d_avg', 'target_price_7d', 'target_price_30d',
    'price_7d_min', 'price_30d_min', 'value_score', 'price_52w_min',
    'price_52w_max', 'price_30d_std', 'price_change_abs',
    'discount_vs_30d_avg', 'price_7d_std', 'popularity_score',
    'price_change_pct', 'price_volatility_30d', 'discount_pct',
    'review_count'
]


# ============================================================
# 2. LEER DATOS
# ============================================================
log("Leyendo datos...")
real_data = pd.read_csv(RUTA_CSV)
log(f"Shape inicial: {real_data.shape}")
log(f"RAM: {real_data.memory_usage(deep=True).sum() / 1e9:.2f} GB")


# ============================================================
# 3. LIMPIAR
# ============================================================
for col in COLUMNAS_A_ELIMINAR:
    if col in real_data.columns:
        real_data = real_data.drop(columns=[col])
        log(f"Eliminada: {col}")

real_data = real_data.fillna(0)
log("Nulos rellenados con 0")
gc.collect()


# ============================================================
# 4. DETECTAR METADATA
# ============================================================
log("Detectando metadata...")
metadata = Metadata.detect_from_dataframe(real_data, table_name='mi_dataset')


# ============================================================
# 5. CORREGIR METADATA
# ============================================================
for col in COLUMNAS_ID:
    if col in real_data.columns:
        metadata.update_column(column_name=col, sdtype='id')
        log(f"'{col}' → id")

for col in COLUMNAS_NUMERICAS_FORZADAS:
    if col in real_data.columns:
        metadata.update_column(column_name=col, sdtype='numerical')
log(f"Forzadas {len(COLUMNAS_NUMERICAS_FORZADAS)} columnas como numéricas")


# ============================================================
# 6. CREAR Y ENTRENAR (GaussianCopula)
# ============================================================
log("Creando sintetizador GaussianCopula...")
synthesizer = GaussianCopulaSynthesizer(
    metadata,
    enforce_min_max_values=True,
    enforce_rounding=True,
    numerical_distributions={
        # Distribuciones por defecto (puedes cambiarlas)
        'price': 'beta',
        'original_price': 'beta',
        'discount_pct': 'beta',
        'avg_rating': 'beta',
        'review_count': 'gamma',
    }
)

log("Iniciando fit...")
synthesizer.fit(real_data)
log("¡Fit terminado!")


# ============================================================
# 7. GENERAR
# ============================================================
log(f"Generando {N_FILAS_SINTETICAS} filas...")
synthetic_data = synthesizer.sample(num_rows=N_FILAS_SINTETICAS)
log(f"Sintético: {synthetic_data.shape}")


# ============================================================
# 8. GUARDAR
# ============================================================
synthetic_data.to_csv(ARCHIVO_SALIDA, index=False)
real_data.to_csv("datos_reales_usados.csv", index=False)
log(f"Guardados: {ARCHIVO_SALIDA} y datos_reales_usados.csv")


# ============================================================
# 9. EVALUACIÓN RÁPIDA DE CALIDAD
# ============================================================
log("Evaluando calidad...")
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(real_data, synthetic_data, metadata)
score = quality_report.get_score()
log(f"Score de calidad: {score:.4f}")
log("Detalle por columna:")
print(quality_report.get_properties())

[17:43:41] Leyendo datos...
[17:43:42] Shape inicial: (200000, 46)
[17:43:42] RAM: 0.17 GB
[17:43:42] Eliminada: timestamp
[17:43:42] Nulos rellenados con 0
[17:43:42] Detectando metadata...
[17:43:43] 'product_id' → id
[17:43:43] 'title' → id
[17:43:43] 'asin' → id
[17:43:43] Forzadas 22 columnas como numéricas
[17:43:43] Creando sintetizador GaussianCopula...
[17:43:43] Iniciando fit...


/home/cristopher/Documentos/Workspace/Deep Learning/AVANCE-DE-DL/venv/lib/python3.12/site-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


[17:46:21] ¡Fit terminado!
[17:46:21] Generando 1000000 filas...
[17:47:33] Sintético: (1000000, 45)
[17:48:03] Guardados: datos_sinteticos_copula.csv y datos_reales_usados.csv
[17:48:03] Evaluando calidad...


/tmp/ipykernel_110137/410521725.py:128: FutureWarning: The evaluation functions are now accessible via the 'sdv.evaluation' module.
  from sdv.evaluation.single_table import evaluate_quality


Generating report ...

(1/4) Evaluating Column Shapes: |██████████| 45/45 [00:06<00:00,  6.53it/s]|
Column Shapes Score: 94.3%

(2/4) Evaluating Column Pair Trends: |██████████| 990/990 [00:47<00:00, 20.76it/s]|
Column Pair Trends Score: 91.67%

(3/4) Evaluating Cardinality: N/A
This property does not apply to single-table data.

(4/4) Evaluating Intertable Trends: N/A
This property does not apply to single-table data.

Overall Score (Average): 92.99%

[17:48:57] Score de calidad: 0.9299
[17:48:57] Detalle por columna:
             Property     Score
0       Column Shapes  0.943022
1  Column Pair Trends  0.916727


### Comparación datos originales V/S sinteticos

In [3]:
# ============================================================
# evaluar_calidad.py — Comparación estadística real vs sintético
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy import stats
from scipy.spatial.distance import jensenshannon
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import rbf_kernel

from sdv.metadata import Metadata
from sdv.evaluation.single_table import evaluate_quality


# ============================================================
# 1. CARGAR DATOS
# ============================================================
print("Cargando datos...")
real = pd.read_csv("datos_reales_usados.csv")
synth = pd.read_csv("datos_sinteticos_copula.csv")

# Asegurar mismas columnas
columnas_comunes = [c for c in real.columns if c in synth.columns]
real = real[columnas_comunes]
synth = synth[columnas_comunes]

print(f"Real: {real.shape}, Sintético: {synth.shape}")


# ============================================================
# 2. IDENTIFICAR TIPOS DE COLUMNAS
# ============================================================
# Excluir IDs (no tiene sentido comparar product_id, title, asin)
cols_excluir = ['product_id', 'title', 'asin']
cols_analizar = [c for c in real.columns if c not in cols_excluir]

# Numéricas
cols_num = real[cols_analizar].select_dtypes(include=[np.number]).columns.tolist()

# Categóricas
cols_cat = real[cols_analizar].select_dtypes(include=['object', 'category']).columns.tolist()

print(f"Numéricas: {len(cols_num)}")
print(f"Categóricas: {len(cols_cat)}")


# ============================================================
# 3. MÉTRICAS UNIVARIADAS — NUMÉRICAS
# ============================================================
print("\n" + "=" * 70)
print("MÉTRICAS UNIVARIADAS - COLUMNAS NUMÉRICAS")
print("=" * 70)

resultados_num = []

for col in cols_num:
    r = real[col].dropna().values
    s = synth[col].dropna().values

    if len(r) < 2 or len(s) < 2:
        continue

    # KS test
    ks_stat, ks_pvalue = stats.ks_2samp(r, s)

    # Wasserstein
    wd = stats.wasserstein_distance(r, s)

    # KL y JS divergence (necesitan histogramas)
    bins = np.histogram_bin_edges(np.concatenate([r, s]), bins=50)
    p, _ = np.histogram(r, bins=bins, density=True)
    q, _ = np.histogram(s, bins=bins, density=True)
    # Evitar ceros
    p = p + 1e-10
    q = q + 1e-10
    p = p / p.sum()
    q = q / q.sum()

    kl = stats.entropy(p, q)
    js = jensenshannon(p, q) ** 2  # jensenshannon devuelve sqrt

    # Medias y std
    mean_diff = abs(r.mean() - s.mean()) / (abs(r.mean()) + 1e-10) * 100
    std_diff = abs(r.std() - s.std()) / (abs(r.std()) + 1e-10) * 100

    resultados_num.append({
        'columna': col,
        'KS': ks_stat,
        'KS_pvalue': ks_pvalue,
        'Wasserstein': wd,
        'KL': kl,
        'JS': js,
        'mean_real': r.mean(),
        'mean_synth': s.mean(),
        'mean_diff_%': mean_diff,
        'std_diff_%': std_diff
    })

df_num = pd.DataFrame(resultados_num)
print(df_num.to_string(index=False))
df_num.to_csv("metricas_numericas.csv", index=False)


# ============================================================
# 4. MÉTRICAS UNIVARIADAS — CATEGÓRICAS
# ============================================================
print("\n" + "=" * 70)
print("MÉTRICAS UNIVARIADAS - COLUMNAS CATEGÓRICAS")
print("=" * 70)

resultados_cat = []

for col in cols_cat:
    r_counts = real[col].value_counts(normalize=True)
    s_counts = synth[col].value_counts(normalize=True)

    # Alinear índices
    todos = r_counts.index.union(s_counts.index)
    p = r_counts.reindex(todos, fill_value=0).values
    q = s_counts.reindex(todos, fill_value=0).values

    # Total Variation Distance
    tvd = 0.5 * np.sum(np.abs(p - q))

    # JS
    js = jensenshannon(p + 1e-10, q + 1e-10) ** 2

    resultados_cat.append({
        'columna': col,
        'n_categorias_real': len(r_counts),
        'n_categorias_synth': len(s_counts),
        'TVD': tvd,
        'JS': js
    })

df_cat = pd.DataFrame(resultados_cat)
print(df_cat.to_string(index=False))
df_cat.to_csv("metricas_categoricas.csv", index=False)


# ============================================================
# 5. CORRELACIONES (BIVARIADA)
# ============================================================
print("\n" + "=" * 70)
print("CORRELACIONES - PEARSON Y SPEARMAN")
print("=" * 70)

# Solo numéricas
real_num = real[cols_num]
synth_num = synth[cols_num]

# Pearson
corr_real_p = real_num.corr(method='pearson')
corr_synth_p = synth_num.corr(method='pearson')
diff_pearson = np.abs(corr_real_p - corr_synth_p)

# Spearman
corr_real_s = real_num.corr(method='spearman')
corr_synth_s = synth_num.corr(method='spearman')
diff_spearman = np.abs(corr_real_s - corr_synth_s)

# Frobenius norm de la diferencia
frob_pearson = np.linalg.norm(diff_pearson.values, 'fro')
frob_spearman = np.linalg.norm(diff_spearman.values, 'fro')

# Promedio de diferencia absoluta (excluyendo diagonal)
mask = ~np.eye(len(cols_num), dtype=bool)
mean_diff_pearson = diff_pearson.values[mask].mean()
mean_diff_spearman = diff_spearman.values[mask].mean()

print(f"Frobenius norm Pearson: {frob_pearson:.4f}")
print(f"Frobenius norm Spearman: {frob_spearman:.4f}")
print(f"Diferencia media Pearson: {mean_diff_pearson:.4f}")
print(f"Diferencia media Spearman: {mean_diff_spearman:.4f}")

# Guardar matrices
corr_real_p.to_csv("corr_real_pearson.csv")
corr_synth_p.to_csv("corr_synth_pearson.csv")
diff_pearson.to_csv("corr_diff_pearson.csv")


# ============================================================
# 6. MMD (MULTIVARIADA)
# ============================================================
print("\n" + "=" * 70)
print("MMD (Maximum Mean Discrepancy)")
print("=" * 70)

def calcular_mmd(X, Y, gamma=1.0, max_samples=5000):
    """MMD con kernel RBF. Usa submuestra para no explotar."""
    if len(X) > max_samples:
        idx = np.random.choice(len(X), max_samples, replace=False)
        X = X[idx]
    if len(Y) > max_samples:
        idx = np.random.choice(len(Y), max_samples, replace=False)
        Y = Y[idx]

    XX = rbf_kernel(X, X, gamma=gamma)
    YY = rbf_kernel(Y, Y, gamma=gamma)
    XY = rbf_kernel(X, Y, gamma=gamma)

    mmd = XX.mean() + YY.mean() - 2 * XY.mean()
    return mmd

# Estandarizar antes de MMD
scaler = StandardScaler()
real_scaled = scaler.fit_transform(real_num.fillna(0))
synth_scaled = scaler.transform(synth_num.fillna(0))

mmd = calcular_mmd(real_scaled, synth_scaled)
print(f"MMD: {mmd:.6f}")

# Guardar
with open("metricas_resumen.txt", "w") as f:
    f.write(f"Frobenius Pearson: {frob_pearson:.4f}\n")
    f.write(f"Frobenius Spearman: {frob_spearman:.4f}\n")
    f.write(f"Diferencia media Pearson: {mean_diff_pearson:.4f}\n")
    f.write(f"Diferencia media Spearman: {mean_diff_spearman:.4f}\n")
    f.write(f"MMD: {mmd:.6f}\n")


# ============================================================
# 7. DCR (PRIVACIDAD)
# ============================================================
print("\n" + "=" * 70)
print("DCR (Distance to Closest Record)")
print("=" * 70)

from sklearn.neighbors import NearestNeighbors

# Submuestra para eficiencia
n_dcr = min(2000, len(real_scaled), len(synth_scaled))
real_dcr = real_scaled[np.random.choice(len(real_scaled), n_dcr, replace=False)]
synth_dcr = synth_scaled[np.random.choice(len(synth_scaled), n_dcr, replace=False)]

nn = NearestNeighbors(n_neighbors=1)
nn.fit(real_dcr)
distancias, _ = nn.kneighbors(synth_dcr)

dcr_media = distancias.mean()
dcr_min = distancias.min()
print(f"DCR media: {dcr_media:.4f}")
print(f"DCR mínima: {dcr_min:.4f}")
print(f"(DCR muy bajo → sintéticos copian reales; muy alto → sintéticos no aprenden nada)")

# Copy ratio
copias_exactas = (distancias < 1e-6).sum()
print(f"Copias exactas: {copias_exactas} de {n_dcr} ({copias_exactas/n_dcr*100:.2f}%)")


# ============================================================
# 8. REPORTE SDV (integrado)
# ============================================================
print("\n" + "=" * 70)
print("REPORTE INTEGRADO SDV")
print("=" * 70)

metadata = Metadata.detect_from_dataframe(real, table_name='t')
for col in cols_excluir:
    if col in real.columns:
        metadata.update_column(column_name=col, sdtype='id')

quality_report = evaluate_quality(real, synth, metadata)
print(f"Score global: {quality_report.get_score():.4f}")
print(quality_report.get_properties())


print("\n" + "=" * 70)
print("LISTO. Archivos generados:")
print("  - metricas_numericas.csv")
print("  - metricas_categoricas.csv")
print("  - corr_real_pearson.csv")
print("  - corr_synth_pearson.csv")
print("  - corr_diff_pearson.csv")
print("  - metricas_resumen.txt")
print("=" * 70)

Cargando datos...
Real: (200000, 45), Sintético: (200000, 45)
Numéricas: 37
Categóricas: 5

MÉTRICAS UNIVARIADAS - COLUMNAS NUMÉRICAS
                  columna       KS     KS_pvalue  Wasserstein           KL           JS   mean_real  mean_synth  mean_diff_%  std_diff_%
                     year 0.000865  9.999993e-01     0.000865 1.496451e-06 3.741130e-07 2023.499005 2023.499870     0.000043    0.000195
                    month 0.162950  0.000000e+00     1.313815 1.426036e-01 2.785602e-02    6.507280    7.821095    20.189926   12.191027
                      day 0.036985 2.760598e-119     0.556070 1.012491e-02 2.634822e-03   15.696070   16.020080     2.064275    5.633332
                dayofweek 0.001740  9.220493e-01     0.004425 1.914788e-05 4.786641e-06    2.995535    2.998330     0.093306    0.141973
                    price 0.061560  0.000000e+00    34.180249 1.971949e-02 4.303396e-03  196.569863  201.507954     2.512130   16.856064
           original_price 0.066180  0.000000

# Evaluación de Calidad: Datos Reales vs Sintéticos

**Objetivo**: Comparar 200,000 muestras reales con 200,000 muestras sintéticas generadas con GaussianCopula, usando métricas estadísticas univariadas, bivariadas y multivariadas.

**Método de generación**: GaussianCopula (SDV)

**Métricas a evaluar**:
- Univariadas numéricas: KS, Wasserstein, KL, JS, mean_diff, std_diff
- Univariadas categóricas: TVD, JS
- Bivariadas: Pearson, Spearman, Frobenius norm
- Multivariadas: MMD
- Privacidad: DCR, Copy Ratio
- Integradas: SDV Quality Score

## 1. Métricas univariadas (por columna)

### 1.1 KS (Kolmogorov-Smirnov)

**Qué mide**: la distancia máxima entre las funciones de distribución acumulada (CDF) de reales y sintéticos.

**Fórmula**: `KS = max|CDF_real(x) - CDF_synth(x)|`

**Rango**: [0, 1]. 0 = idénticas, 1 = completamente separadas.

| Valor | Interpretación |
|-------|----------------|
| < 0.05 | ✅ Excelente |
| 0.05 - 0.10 | ✅ Bueno |
| 0.10 - 0.20 | 🟡 Aceptable |
| > 0.20 | 🔴 Malo |

**Nota**: KS no indica el signo de la diferencia, solo la magnitud.

### 1.2 KS p-value

**Qué mide**: probabilidad de observar esa diferencia si ambas muestras vinieran de la misma distribución.

- p > 0.05: no hay evidencia de diferencia
- p < 0.05: diferencia estadísticamente significativa

**⚠️ Cuidado**: con n=200,000 el test es muy sensible y detecta diferencias mínimas como significativas. **Prioriza el KS sobre el p-value**.

### 1.3 Wasserstein (Earth Mover's Distance)

**Qué mide**: el "costo" de transformar la distribución sintética en la real.

**Rango**: [0, ∞). Depende de la escala de la variable.

**Uso**: útil para comparar entre columnas o entre modelos. Normalizar por `std_real` para comparar entre variables.

### 1.4 KL (Kullback-Leibler Divergence)

**Qué mide**: información perdida al usar la sintética para aproximar la real.

**Fórmula**: `KL(P||Q) = Σ P(x)·log(P(x)/Q(x))`

**Rango**: [0, ∞). No simétrica, no es distancia.

| Valor | Interpretación |
|-------|----------------|
| < 0.01 | ✅ Excelente |
| 0.01 - 0.05 | ✅ Bueno |
| 0.05 - 0.20 | 🟡 Aceptable |
| > 0.20 | 🔴 Malo |

### 1.5 JS (Jensen-Shannon Divergence)

**Qué mide**: versión simétrica y acotada de KL.

**Rango**: [0, 1] (o [0, 0.69] con log natural).

| Valor | Interpretación |
|-------|----------------|
| < 0.01 | ✅ Excelente |
| 0.01 - 0.05 | ✅ Bueno |
| 0.05 - 0.10 | 🟡 Aceptable |
| > 0.10 | 🔴 Malo |

**Ventaja**: no explota con probabilidades 0. Mejor que KL para reportar.

### 1.6 mean_diff_% y std_diff_%

**Qué miden**: diferencia porcentual de la media y desviación estándar entre reales y sintéticos.

| Diferencia | Interpretación |
|-----------|----------------|
| < 1% | ✅ Excelente |
| 1 - 5% | ✅ Bueno |
| 5 - 15% | 🟡 Aceptable |
| > 15% | 🔴 Malo |

### 1.7 TVD (Total Variation Distance) — categóricas

**Qué mide**: mitad de la suma de diferencias absolutas entre frecuencias relativas.

**Fórmula**: `TVD = 0.5 · Σ|p_real(c) - p_synth(c)|`

**Rango**: [0, 1].

| Valor | Interpretación |
|-------|----------------|
| < 0.05 | ✅ Excelente |
| 0.05 - 0.10 | ✅ Bueno |
| 0.10 - 0.20 | 🟡 Aceptable |
| > 0.20 | 🔴 Malo |

In [4]:
# ============================================================
# tablas_a_imagenes.py
# Convierte los CSV de métricas en imágenes PNG
# ============================================================
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['font.size'] = 10


def tabla_a_imagen(df, titulo, ruta_salida, ancho=14, alto_por_fila=0.4):
    """Convierte un DataFrame en una imagen PNG."""
    n_filas = len(df)
    alto = max(3, n_filas * alto_por_fila + 1.5)

    fig, ax = plt.subplots(figsize=(ancho, alto))
    ax.axis('tight')
    ax.axis('off')

    # Redondear floats para que no se vean largos
    df_mostrar = df.copy()
    for col in df_mostrar.select_dtypes(include='float').columns:
        df_mostrar[col] = df_mostrar[col].round(4)

    tabla = ax.table(
        cellText=df_mostrar.values,
        colLabels=df_mostrar.columns,
        cellLoc='center',
        loc='center'
    )
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    tabla.scale(1, 1.5)

    # Colorear encabezado
    for i in range(len(df_mostrar.columns)):
        tabla[(0, i)].set_facecolor('#4472C4')
        tabla[(0, i)].set_text_props(weight='bold', color='white')

    # Colorear filas alternas
    for i in range(1, len(df_mostrar) + 1):
        for j in range(len(df_mostrar.columns)):
            if i % 2 == 0:
                tabla[(i, j)].set_facecolor('#F2F2F2')

    ax.set_title(titulo, fontsize=13, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(ruta_salida, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Guardado: {ruta_salida}")


# ============================================================
# Convertir cada CSV a imagen
# ============================================================

# 1. Métricas numéricas
df = pd.read_csv("metricas_numericas.csv")
tabla_a_imagen(df, "Métricas univariadas — Columnas numéricas",
               "img_metricas_numericas.png", ancho=16, alto_por_fila=0.35)

# 2. Métricas categóricas
df = pd.read_csv("metricas_categoricas.csv")
tabla_a_imagen(df, "Métricas univariadas — Columnas categóricas",
               "img_metricas_categoricas.png", ancho=12, alto_por_fila=0.5)

# 3. Resumen general (si existe el txt)
try:
    with open("metricas_resumen.txt") as f:
        lineas = [l.strip().split(": ") for l in f if ": " in l]
    df_resumen = pd.DataFrame(lineas, columns=["Métrica", "Valor"])
    tabla_a_imagen(df_resumen, "Resumen de métricas multivariadas",
                   "img_resumen.png", ancho=10, alto_por_fila=0.6)
except FileNotFoundError:
    print("No existe metricas_resumen.txt, saltando")

# 4. Top 10 diferencias de correlación
try:
    diff = pd.read_csv("corr_diff_pearson.csv", index_col=0)
    # Convertir matriz a lista de pares
    pares = []
    for i, c1 in enumerate(diff.columns):
        for c2 in diff.columns[i+1:]:
            pares.append({"Par": f"{c1} vs {c2}", "Diferencia": diff.loc[c1, c2]})
    df_pares = pd.DataFrame(pares).sort_values("Diferencia", ascending=False).head(15)
    tabla_a_imagen(df_pares, "Top 15 pares con mayor diferencia de correlación (Pearson)",
                   "img_top_diferencias_corr.png", ancho=10, alto_por_fila=0.4)
except FileNotFoundError:
    print("No existe corr_diff_pearson.csv, saltando")

print("\nListo. Revisa los archivos img_*.png")

Guardado: img_metricas_numericas.png
Guardado: img_metricas_categoricas.png
Guardado: img_resumen.png
Guardado: img_top_diferencias_corr.png

Listo. Revisa los archivos img_*.png


## Bibliografía

1. https://research.itu.edu.tr/en/publications/investigating-tabular-generative-models-for-synthetic-data-genera/#1